# Zalando SE — ESG data exploration (SAM 503 · Lab 4)
Load, clean and explore the 10-year dataset before building the Streamlit dashboard.

**Step 1 — set your GitHub repo below** (after you push this project).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
print('Hello from Colab!')

# Option A: read from your GitHub repo (replace USER)
BASE = 'https://raw.githubusercontent.com/USER/sam_503_esg-lab/main/data/'
# Option B: upload the data/ folder to Colab and use BASE = 'data/'
try:
    ts = pd.read_csv(BASE + 'zalando_timeseries.csv')
except Exception:
    BASE = 'data/'
    ts = pd.read_csv(BASE + 'zalando_timeseries.csv')
ts.head(10)

## Data cleaning & checks

In [ ]:
print(ts.isna().sum())
# Methodology era flag — never compare GHG across this break
ts['era'] = ts.year.apply(lambda y: 'ESRS/restated' if y >= 2024 else ('legacy' if y >= 2017 else 'no GHG data'))
# Check: 2025 total must equal AR2025 reported 2,540,750.6 tCO2e
assert abs(ts.loc[ts.year==2025,'total_mb_t'].item() - 2540750.6) < 1
ts[['year','era','scope1_t','scope2_mb_t','scope3_t','total_mb_t','intensity_t_per_m_eur']]

## Exploration

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ts.year, ts.scope1_t, marker='o', label='Scope 1')
ax[0].plot(ts.year, ts.scope2_mb_t, marker='o', label='Scope 2 (MB)')
ax[0].axvline(2023.5, ls=':', c='grey'); ax[0].set_title('Scope 1 & 2 (tCO2e)'); ax[0].legend()
ax[1].bar(ts.year, ts.scope3_t/1e6, color='#1baf7a')
ax[1].axvline(2023.5, ls=':', c='grey'); ax[1].set_title('Scope 3 (Mt CO2e) — note restatement')
plt.tight_layout(); plt.show()

In [ ]:
s3 = pd.read_csv(BASE + 'zalando_scope3_categories.csv')
s3['share_2025'] = s3.t_2025 / s3.t_2025.sum()
s3.sort_values('t_2025', ascending=False)

In [ ]:
# Simple target check (Python building blocks: variables, conditions, functions)
def on_track(achieved, target, base_year, target_year, year=2025):
    expected = target * (year - base_year) / (target_year - base_year)
    return achieved >= expected, round(expected*100, 1)

for name, ach, tgt in [('Scope 1+2', 0.170, 0.55), ('Scope 3 cat.1+4', 0.106, 0.33)]:
    ok, exp = on_track(ach, tgt, 2022, 2033)
    print(f"{name}: achieved {ach*100:.1f}% vs linear {exp}% ->", 'ON TRACK' if ok else 'BEHIND')

## Launch the Streamlit app from Colab (optional)

In [ ]:
!pip -q install streamlit plotly
# app.py and data/ must be in the Colab working directory (clone your repo):
# !git clone https://github.com/USER/sam_503_esg-lab.git && %cd sam_503_esg-lab
!streamlit run app.py &>/content/log.txt &
!npx --yes localtunnel --port 8501